In [39]:
import fitz
import datetime
import re


In [2]:
file_path = "/home/stark/Documents/Research Pappers/Attention_is_All_you_need.pdf"



In [3]:
docs = fitz.open(file_path)

In [4]:
docs

Document('/home/stark/Documents/Research Pappers/Attention_is_All_you_need.pdf')

In [5]:


word_id = 0
pages_data = []

for page_num, page in enumerate(docs):
    words = page.get_text("words")
    
    page_words = []
    for word in words:
        page_words.append({
            "word_id": word_id,
            "text":    word[4],
            "coords":  (word[0], word[1], word[2], word[3]),
            "page":    page_num
        })
        word_id += 1
    
    page_text = " ".join([w["text"] for w in page_words])
    
    pages_data.append({
        "page_num":  page_num,
        "words":     page_words,
        "full_text": page_text
    })




In [15]:
def preprocess_pages_data(pages_data):
    skip = {".", ",", "!", "?", ";", ":", "-", "--", "•", "*"}
    
    for page in pages_data:
        page["words"] = [
            w for w in page["words"]
            if w["text"].strip() not in skip
        ]
        page["full_text"] = " ".join(w["text"] for w in page["words"])
    
    return pages_data

pages_data = preprocess_pages_data(pages_data)

In [46]:
pages_data

[{'page_num': 0,
  'words': [{'word_id': 0,
    'text': 'Provided',
    'coords': (124.66600036621094,
     72.89631652832031,
     167.6569061279297,
     84.85151672363281),
    'page': 0},
   {'word_id': 1,
    'text': 'proper',
    'coords': (170.6457061767578,
     72.89631652832031,
     201.84877014160156,
     84.85151672363281),
    'page': 0},
   {'word_id': 2,
    'text': 'attribution',
    'coords': (204.8375701904297,
     72.89631652832031,
     254.41578674316406,
     84.85151672363281),
    'page': 0},
   {'word_id': 3,
    'text': 'is',
    'coords': (257.40460205078125,
     72.89631652832031,
     265.37872314453125,
     84.85151672363281),
    'page': 0},
   {'word_id': 4,
    'text': 'provided,',
    'coords': (268.3675231933594,
     72.89631652832031,
     313.677734375,
     84.85151672363281),
    'page': 0},
   {'word_id': 5,
    'text': 'Google',
    'coords': (316.6665344238281,
     72.89631652832031,
     351.8626403808594,
     84.85151672363281),
    '

In [43]:
len_words = 150
all_chunks = []

for page in pages_data:
    current_chunk_words = []
    
    for word in page["words"]:
        current_chunk_words.append(word)
        # save chunk when sentence ends AND chunk is long enough
        if word["text"].endswith((".", "?", "!")) and len(current_chunk_words) >= len_words:
            all_chunks.append({
                "text":     " ".join(w["text"] for w in current_chunk_words),
                "words":    current_chunk_words,
                "start_id": current_chunk_words[0]["word_id"],
                "end_id":   current_chunk_words[-1]["word_id"],
                "page":     page["page_num"],
                "coords":   [w["coords"] for w in current_chunk_words]
            })
            current_chunk_words = []  # reset after saving
    
    # save leftover words at end of page
    if current_chunk_words:
        all_chunks.append({
            "text":     clean_text(" ".join(w["text"] for w in current_chunk_words)),
            "words":    current_chunk_words,
            "start_id": current_chunk_words[0]["word_id"],
            "end_id":   current_chunk_words[-1]["word_id"],
            "page":     page["page_num"],
            "coords":   [w["coords"] for w in current_chunk_words]
        })

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 44


In [49]:

for chunks in all_chunks:
    
    print(chunks["text"])
    print("----------------------------------------------------------------------------")

Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Exper

In [42]:
def clean_text(text):
    # remove citation numbers like [13], [2, 5]
    text = re.sub(r'\[\d+(?:,\s*\d+)*\]', '', text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [45]:
all_chunks

[{'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions enti